In [ ]:
import os
import rasterio
import numpy as np

def pad_image(image, target_height, target_width):
    padded_image = np.zeros((image.shape[0], target_height, target_width), dtype=image.dtype)
    padded_image[:, :image.shape[1], :image.shape[2]] = image
    return padded_image

def stack_images(folder1, folder2, destination_folder):
    os.makedirs(destination_folder, exist_ok=True)

    # List files in both folders (assuming same filenames in both)
    images1 = sorted([f for f in os.listdir(folder1) if f.endswith('.tif')])
    images2 = sorted([f for f in os.listdir(folder2) if f.endswith('.tif')])

    # Create a mapping to match files based on their identifier (e.g., _1001, _1002)
    image_map1 = {f.split('_')[-1]: f for f in images1}
    image_map2 = {f.split('_')[-1]: f for f in images2}

    # Identify common files based on the identifier
    common_identifiers = set(image_map1.keys()) & set(image_map2.keys())

    if not common_identifiers:
        print("No matching images found based on identifiers.")
        return

    for identifier in common_identifiers:
        img1_path = os.path.join(folder1, image_map1[identifier])
        img2_path = os.path.join(folder2, image_map2[identifier])

        # Read images
        with rasterio.open(img1_path) as src1, rasterio.open(img2_path) as src2:
            # Determine target dimensions
            target_height = max(src1.height, src2.height)
            target_width = max(src1.width, src2.width)

            # Read and pad images to match target dimensions
            img1_data = pad_image(src1.read(), target_height, target_width)
            img2_data = pad_image(src2.read(), target_height, target_width)

            # Stack images (early date folder images first)
            stacked_data = np.concatenate([img1_data, img2_data], axis=0)

            # Construct output filename based on original image name
            #output_name = f"{os.path.splitext(image_map1[identifier])[0]}_stacked.tif"
            output_name = f"{os.path.splitext(image_map1[identifier])[0]}.tif"
            stacked_path = os.path.join(destination_folder, output_name)

            # Write stacked image to destination folder
            with rasterio.open(
                stacked_path, 'w',
                driver='GTiff',
                height=stacked_data.shape[1],
                width=stacked_data.shape[2],
                count=stacked_data.shape[0],
                dtype=stacked_data.dtype,
                crs=src1.crs,
                transform=src1.transform
            ) as dst:
                dst.write(stacked_data)

            print(f"Stacked image saved: {stacked_path}")

# Example usage
folder1 = r"D:\KOUSHAL\practice\20220518"
folder2 = r"D:\KOUSHAL\practice\20220603"
destination_folder = r"D:\practice\stacked_images"
stack_images(folder1, folder2, destination_folder)
